<a href="https://colab.research.google.com/github/Rflavia/mineracao_dados/blob/main/03_regras_associacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 3 — Regras de Associação: Candidatos 2026 (TSE)

## Pergunta de negócio

> **Que combinações de características dos candidatos aparecem juntas com mais frequência do que se fossem independentes? Em particular: quais atributos (partido, gênero, escolaridade, cor/raça, UF de nascimento) estão associados a cada perfil identificado na Fase 2 — e, assim que a eleição ocorrer, a ser eleito?**

Até aqui, a clusterização (Fase 2) agrupou candidatos por **distância** em variáveis numéricas (idade, escolaridade, patrimônio). Regras de associação fazem uma pergunta diferente: em vez de "quem está perto de quem", **quais combinações de atributos categóricos aparecem juntas com mais frequência do que o acaso explicaria**? A técnica clássica pra isso é o **Apriori**, que originalmente resolve o problema da "cesta de compras" (que produtos são comprados juntos) — aqui, cada candidato é uma cesta, e cada valor de cada variável categórica (partido, gênero, escolaridade...) é um item dessa cesta.


### Carregamento do repositório
A célula abaixo prepara o repositório no ambiente temporário do Google Colab. Caso o repositório ainda não esteja disponível no ambiente, ele será clonado do GitHub. Se a pasta mineracao_dados já existir, o clone não será realizado novamente. Em seguida, o diretório de trabalho é definido e a pasta dados é disponibilizada por meio do sparse checkout.

In [1]:
import os
import sys
#Nesta fase, vamos usar o repositório mineracao_dados para acessar os dados e scripts necessários.

# Só no Google Colab: localmente os arquivos já estão na pasta do projeto
# (e o sparse-checkout abaixo alteraria o repositório local)
if "google.colab" in sys.modules:
    if not os.path.exists('/content/mineracao_dados'):
        !git clone --depth 1 --filter=blob:none --sparse https://github.com/Rflavia/mineracao_dados.git
    else:
        print("Repositório já existe no ambiente.")

    %cd /content/mineracao_dados
    !git sparse-checkout set dados

## Configuração

Use os **mesmos valores** que você usou nas Fases 0/1/2, para carregar o arquivo certo.


In [2]:
CARGO = "DEPUTADO FEDERAL"
UFS_REGIAO = ["PR", "RS" , "SC"]
ANO_ELEICAO = 2026


## Preparando a base — reconstituindo os clusters nomeados da Fase 2

Apriori precisa de **itens categóricos**, não de distância — mas queremos incluir o cluster da Fase 2 como mais um item da cesta (pra achar regras do tipo "esse perfil de candidato é do cluster X"). O resultado oficial da Fase 2 é o **Bisecting K-Means com as 4 variáveis** (`IDADE`, `ANOS_ESTUDO`, `Total_Bens_Log`, `Total_Gasto_Campanha_Log`, padronizadas com `StandardScaler`), **k=4**, critério de inércia média — ver `02_clusterizacao_com_despesa.ipynb`, Parte 3. Os quatro clusters foram batizados a partir da ficha técnica e da composição demográfica/partidária:

- **A — Estabelecidos com patrimônio e campanha** (712, 64%): declaram bens e já têm gasto de campanha.
- **B — Em campanha sem patrimônio declarado** (222, 20%): todos já gastam, mas 87% não declararam bens.
- **C — Nível superior sem campanha** (94, 8%): 95% sem gasto; 72% com superior completo ou incompleto; o grupo mais jovem.
- **D — Base popular sem campanha** (89, 8%): a menor escolaridade (8,6 anos) e o grupo mais velho, 89% sem gasto.

Recalculamos aqui, rapidamente, a mesma partição (mesma `SEMENTE`, mesmo critério, mesmos nomes), para este notebook rodar sozinho.

In [3]:
import warnings
warnings.filterwarnings("ignore")
if "google.colab" in sys.modules:
    !pip install -q pyvis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from mlxtend.frequent_patterns import apriori, association_rules
from pyvis.network import Network
from IPython.display import IFrame

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)  # não truncar antecedents/consequents nas tabelas

SEMENTE = 42
np.random.seed(SEMENTE)  # mesmo racional de reprodutibilidade defensiva dos notebooks anteriores

if UFS_REGIAO is None:
    nome_uf = "BRASIL"
elif isinstance(UFS_REGIAO, list):
    nome_uf = "_".join(UFS_REGIAO)
else:
    nome_uf = UFS_REGIAO



nome_cargo = CARGO.replace(' ', '_')
arquivo = f"dados/candidatos_{nome_cargo}_{nome_uf}_{ANO_ELEICAO}_com_despesas.csv"

df = pd.read_csv(arquivo)
print(f"Carregado: {arquivo}  ->  {df.shape[0]} candidatos, {df.shape[1]} colunas")
df.head()

Carregado: dados/candidatos_DEPUTADO_FEDERAL_PR_RS_SC_2026_com_despesas.csv  ->  1117 candidatos, 63 colunas


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,SG_UF,SG_UE,NM_UE,CD_CARGO,DS_CARGO,SQ_CANDIDATO,NR_CANDIDATO,NM_CANDIDATO,NM_URNA_CANDIDATO,NM_SOCIAL_CANDIDATO,NR_CPF_CANDIDATO,DS_EMAIL,CD_SITUACAO_CANDIDATURA,DS_SITUACAO_CANDIDATURA,TP_AGREMIACAO,NR_PARTIDO,SG_PARTIDO,NM_PARTIDO,NR_FEDERACAO,NM_FEDERACAO,SG_FEDERACAO,DS_COMPOSICAO_FEDERACAO,SQ_COLIGACAO,NM_COLIGACAO,DS_COMPOSICAO_COLIGACAO,SG_UF_NASCIMENTO,DT_NASCIMENTO,NR_TITULO_ELEITORAL_CANDIDATO,CD_GENERO,DS_GENERO,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO,BENS IMÓVEIS,BENS MÓVEIS,DINHEIRO,INVESTIMENTOS RENDA FIXA,INVESTIMENTOS RENDA VARIÁVEL,OUTROS BENS,Total_Bens,Total_Bens_Log,IDADE,Total_Gasto_Campanha,QTD_DESPESAS_CONTRATADAS,TEM_DESPESA_CONTRATADA,Total_Gasto_Campanha_Log
0,19/09/2026,16:30:44,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,PR,PR,PARANÁ,6,DEPUTADO FEDERAL,160002532588,4050,MARIO SETO TAKEGUMA JUNIOR,MARIO CERTO,#NULO,6434321910,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,40,PSB,PARTIDO SOCIALISTA BRASILEIRO,-1,#NULO,#NULO,#NULO,160001800094,PARTIDO ISOLADO,PSB,PR,1987-01-07,92467190698,2,MASCULINO,8,SUPERIOR COMPLETO,1,SOLTEIRO(A),4,AMARELA,132,PSICÓLOGO,-1,#NULO,0.0,45000.0,11544.24,87605.86,0.0,52262.32,196412.42,12.187977,39,34140.00,16,True,10.438254
1,19/09/2026,16:30:44,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,PR,PR,PARANÁ,6,DEPUTADO FEDERAL,160002532589,4045,ANTÔNIO FERNANDO TEIGAO,FERNANDÃO,#NULO,30357934920,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,40,PSB,PARTIDO SOCIALISTA BRASILEIRO,-1,#NULO,#NULO,#NULO,160001800094,PARTIDO ISOLADO,PSB,PR,1957-10-21,12790340663,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,257,EMPRESÁRIO,-1,#NULO,0.0,87000.0,4984.47,25453.75,15000.0,227871.27,360309.49,12.794721,69,59862.86,35,True,10.999828
2,19/09/2026,16:30:44,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,PR,PR,PARANÁ,6,DEPUTADO FEDERAL,160002532590,4000,CAMILLA DE MORAES GONDA,CAMILLA GONDA,#NULO,10555616924,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,40,PSB,PARTIDO SOCIALISTA BRASILEIRO,-1,#NULO,#NULO,#NULO,160001800094,PARTIDO ISOLADO,PSB,PR,2000-10-14,115434590639,4,FEMININO,8,SUPERIOR COMPLETO,1,SOLTEIRO(A),1,BRANCA,278,VEREADOR,-1,#NULO,0.0,0.0,0.00,14187.66,0.0,0.00,14187.66,9.560198,25,283738.41,59,True,12.555812
3,19/09/2026,16:30:44,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,PR,PR,PARANÁ,6,DEPUTADO FEDERAL,160002532591,4070,CHARLESTON ROBERTO DE OLIVEIRA MAYER JUNIOR,CHARLESTON JÚNIOR INCLUSÃO,#NULO,10945908946,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,40,PSB,PARTIDO SOCIALISTA BRASILEIRO,-1,#NULO,#NULO,#NULO,160001800094,PARTIDO ISOLADO,PSB,PR,2004-07-05,124740900612,2,MASCULINO,7,SUPERIOR INCOMPLETO,1,SOLTEIRO(A),1,BRANCA,931,"ESTUDANTE, BOLSISTA, ESTAGIÁRIO E ASSEMELHADOS",-1,#NULO,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.000000,22,0.00,1,True,0.000000
4,19/09/2026,16:30:44,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,PR,PR,PARANÁ,6,DEPUTADO FEDERAL,160002532592,4077,OMAR RAIMUNDO PICHETH NETO,OMAR PICHETH,#NULO,82002045968,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,40,PSB,PARTIDO SOCIALISTA BRASILEIRO,-1,#NULO,#NULO,#NULO,160001800094,PARTIDO ISOLADO,PSB,PR,1971-07-28,46300400663,2,MASCULINO,8,SUPERIOR COMPLETO,9,DIVORCIADO(A),1,BRANCA,278,VEREADOR,-1,#NULO,275000.0,40000.0,0.00,19686.98,0.0,0.00,334686.98,12.720954,55,144017.53,48,True,11.877697


In [4]:
ANOS_ESTUDO_POR_GRAU = {
    'ANALFABETO': 0,
    'LÊ E ESCREVE': 1,
    'ENSINO FUNDAMENTAL INCOMPLETO': 4,
    'ENSINO FUNDAMENTAL COMPLETO': 8,
    'ENSINO MÉDIO INCOMPLETO': 9,
    'ENSINO MÉDIO COMPLETO': 11,
    'SUPERIOR INCOMPLETO': 13,
    'SUPERIOR COMPLETO': 16,
    # 'NÃO DIVULGÁVEL' fica de fora de propósito — mesma decisão da Versão 2 do notebook de clusterização
}

df['ANOS_ESTUDO'] = df['DS_GRAU_INSTRUCAO'].str.strip().str.upper().map(ANOS_ESTUDO_POR_GRAU)

In [5]:
# mesmas 4 variáveis e mesma padronização do 02_clusterizacao_com_despesa.ipynb (Parte 3)
colunas_driver = ['IDADE', 'ANOS_ESTUDO', 'Total_Bens_Log', 'Total_Gasto_Campanha_Log']

df_cluster = df.dropna(subset=colunas_driver).copy()
print(f"Base para clusterização/regras: {df_cluster.shape[0]} candidatos "
      f"(descartados {df.shape[0] - df_cluster.shape[0]} sem alguma das 4 variáveis)")

scaler = StandardScaler()
X = scaler.fit_transform(df_cluster[colunas_driver])

Base para clusterização/regras: 1117 candidatos (descartados 0 sem alguma das 4 variáveis)


In [6]:
def inercia(indices):
    if len(indices) < 2:
        return 0.0
    pontos = X[indices]
    centro = pontos.mean(axis=0)
    return float(((pontos - centro) ** 2).sum(axis=1).mean())


def bisecting_kmeans_ate_k(X, k_max, random_state=SEMENTE):
    """Mesma lógica do 02_clusterizacao_com_despesa.ipynb (Parte 3, critério de inércia média) — mas só
    devolve a partição final em k_max clusters, sem guardar o histórico de divisões (essa
    exploração já foi feita e documentada lá)."""
    clusters = {0: np.arange(X.shape[0])}
    proximo_id = 1
    for _ in range(1, k_max):
        id_escolhido = max(clusters, key=lambda cid: inercia(clusters[cid]))
        indices_pai = clusters[id_escolhido]
        if len(indices_pai) < 2:
            break
        km2 = KMeans(n_clusters=2, random_state=random_state, n_init=10)
        labels2 = km2.fit_predict(X[indices_pai])
        id_a, id_b = proximo_id, proximo_id + 1
        proximo_id += 2
        del clusters[id_escolhido]
        clusters[id_a] = indices_pai[labels2 == 0]
        clusters[id_b] = indices_pai[labels2 == 1]
    return clusters


k_bisecting = 4  # mesmo k escolhido no 02_clusterizacao_com_despesa.ipynb (Parte 3)

clusters_finais = bisecting_kmeans_ate_k(X, k_bisecting)
ids_por_tamanho = sorted(clusters_finais, key=lambda cid: len(clusters_finais[cid]), reverse=True)

rotulos = np.empty(X.shape[0], dtype=object)
for letra, cid in zip([chr(65 + i) for i in range(len(ids_por_tamanho))], ids_por_tamanho):
    rotulos[clusters_finais[cid]] = letra
df_cluster['cluster_bisecting'] = rotulos

# mesmos nomes definidos no 02_clusterizacao_com_despesa.ipynb (Parte 3), a partir da mesma ficha técnica
NOMES_CLUSTER_BISECTING = {
    'A': 'Estabelecidos com patrimônio e campanha',
    'B': 'Em campanha sem patrimônio declarado',
    'C': 'Nível superior sem campanha',
    'D': 'Base popular sem campanha',
}
df_cluster['cluster'] = df_cluster['cluster_bisecting'].map(NOMES_CLUSTER_BISECTING)

df_cluster['cluster'].value_counts().sort_index()

cluster
Base popular sem campanha                   89
Em campanha sem patrimônio declarado       222
Estabelecidos com patrimônio e campanha    712
Nível superior sem campanha                 94
Name: count, dtype: int64

### Faixas por quartil das 4 variáveis contínuas

O Apriori só aceita itens categóricos, então cada variável contínua do modelo vira uma faixa por quartil (mesma regra da Parte 2 do `02_clusterizacao_com_despesa.ipynb`: `pd.qcut(q=4, duplicates='drop')`). Quando muitos candidatos têm o mesmo valor, alguns limites de quartil coincidem e a variável fica com menos de 4 faixas:

- `ANOS_ESTUDO` fica com **2 faixas**, porque 61% dos candidatos têm exatamente 16 anos (superior completo);
- `Total_Bens_Log` fica com **3 faixas**, porque 27% declararam patrimônio zero (a primeira faixa junta os zeros com quem tem até a mediana).

Para facilitar a leitura das regras, o nome de cada faixa traz o intervalo em unidades naturais (anos ou R$), e não em log.

In [7]:
def formatar_reais(valor):
    if valor < 1_000:
        return f"R$ {valor:,.0f}"
    if valor < 1_000_000:
        return f"R$ {valor / 1_000:,.0f} mil"
    return f"R$ {valor / 1_000_000:,.1f} mi"


variaveis_faixa = {
    'IDADE': lambda v: f"{v:.0f}",
    'ANOS_ESTUDO': lambda v: f"{v:.0f}",
    'Total_Bens_Log': lambda v: formatar_reais(np.expm1(v)),            # volta do log para R$
    'Total_Gasto_Campanha_Log': lambda v: formatar_reais(np.expm1(v)),
}

for variavel, fmt in variaveis_faixa.items():
    codigos, limites = pd.qcut(df_cluster[variavel], q=4, labels=False, retbins=True, duplicates='drop')
    rotulos_faixa = {
        i: f"Q{i + 1} ({fmt(limites[i])} a {fmt(limites[i + 1])})"
        for i in range(len(limites) - 1)
    }
    df_cluster[f'{variavel}_Faixa'] = codigos.map(rotulos_faixa)

colunas_faixa = [f'{v}_Faixa' for v in variaveis_faixa]

for col in colunas_faixa:
    print(f"--- {col} ---")
    print(df_cluster[col].value_counts().sort_index().to_string())
    print()

--- IDADE_Faixa ---
IDADE_Faixa
Q1 (21 a 40)    302
Q2 (40 a 48)    278
Q3 (48 a 58)    272
Q4 (58 a 85)    265

--- ANOS_ESTUDO_Faixa ---
ANOS_ESTUDO_Faixa
Q1 (4 a 13)     437
Q2 (13 a 16)    680

--- Total_Bens_Log_Faixa ---
Total_Bens_Log_Faixa
Q1 (R$ 0 a R$ 158 mil)          559
Q2 (R$ 158 mil a R$ 710 mil)    280
Q3 (R$ 710 mil a R$ 75.7 mi)    278

--- Total_Gasto_Campanha_Log_Faixa ---
Total_Gasto_Campanha_Log_Faixa
Q1 (R$ 0 a R$ 3 mil)           280
Q2 (R$ 3 mil a R$ 34 mil)      279
Q3 (R$ 34 mil a R$ 144 mil)    279
Q4 (R$ 144 mil a R$ 3.0 mi)    279



## Montando a "cesta" de itens

Apriori não trabalha com números contínuos como o K-Means — ele precisa de **itens binários** (presente/ausente), do mesmo jeito que "pão" e "leite" são itens numa cesta de supermercado. Cada linha vira uma "cesta" (um candidato), e cada valor de cada variável categórica vira um "item" via one-hot encoding (`SG_PARTIDO=PT`, `DS_GENERO=FEMININO`, `cluster=Base popular sem campanha`, ...).

Variáveis escolhidas:
- `SG_PARTIDO`, `DS_GENERO`, `DS_GRAU_INSTRUCAO`, `DS_ESTADO_CIVIL`, `DS_COR_RACA`, `SG_UF_NASCIMENTO` — perfil do candidato.
- `IDADE_Faixa`, `ANOS_ESTUDO_Faixa`, `Total_Bens_Log_Faixa`, `Total_Gasto_Campanha_Log_Faixa` — as 4 variáveis contínuas da clusterização, em faixas por quartil (célula anterior).
- `cluster` — o nome do cluster da Fase 2 (Bisecting K-Means, 4 variáveis, k=4), recalculado acima. Incluir o cluster como item deixa a pergunta "o que caracteriza cada cluster?" (já respondida via ficha técnica na Fase 2) ser respondida de novo do ponto de vista de regras — e permite achar combinações de 2+ variáveis que sozinhas não bastariam.
- `SG_UF` — entra porque o recorte tem 3 UFs (PR, RS e SC); com uma UF fixa ela teria um único valor e não discriminaria nada.
- `DS_SIT_TOT_TURNO` (eleito ou não) — só entra depois que a eleição de fato ocorrer (a coluna hoje vem toda com o mesmo valor, `#NULO`). A seção que usa isso fica pronta, mas roda condicionalmente lá na frente.

**Redundância conhecida:** `ANOS_ESTUDO_Faixa` é derivada de `DS_GRAU_INSTRUCAO` (a faixa Q2 é exatamente "superior completo"). Mantemos as duas — o grau de instrução é mais detalhado, a faixa é a variável do modelo —, mas descartamos as regras que misturam as duas, porque seriam verdadeiras por definição (confiança 1) e não diriam nada.

`DS_OCUPACAO` fica de fora por ora: são 109 ocupações diferentes para 1.117 candidatos, e a maioria teria poucos candidatos, suporte baixo demais para dizer qualquer coisa. O cruzamento de profissão com os clusters está no `02_clusterizacao_com_despesa.ipynb` (top 5 ocupações por cluster).

In [8]:
colunas_apriori = ['SG_PARTIDO', 'DS_GENERO', 'DS_GRAU_INSTRUCAO', 'DS_ESTADO_CIVIL',
                   'DS_COR_RACA', 'SG_UF_NASCIMENTO'] + colunas_faixa + ['cluster']

if UFS_REGIAO is None or len(UFS_REGIAO) > 1:
    colunas_apriori.append('SG_UF')

# a eleição de 2026 ainda não ocorreu -> DS_SIT_TOT_TURNO vem toda com o mesmo valor por enquanto
usar_resultado_eleicao = df_cluster['DS_SIT_TOT_TURNO'].nunique() > 1
if usar_resultado_eleicao:
    colunas_apriori.append('DS_SIT_TOT_TURNO')

cesta_base = df_cluster[['SQ_CANDIDATO'] + colunas_apriori].copy()
cesta = pd.get_dummies(cesta_base, columns=colunas_apriori, prefix_sep='=')
cesta = cesta.set_index('SQ_CANDIDATO')

print(f"{cesta.shape[0]} candidatos (cestas) x {cesta.shape[1]} itens possíveis (um por valor de cada variável)")
cesta_base.head()

1117 candidatos (cestas) x 84 itens possíveis (um por valor de cada variável)


,SQ_CANDIDATO,SG_PARTIDO,DS_GENERO,DS_GRAU_INSTRUCAO,DS_ESTADO_CIVIL,DS_COR_RACA,SG_UF_NASCIMENTO,IDADE_Faixa,ANOS_ESTUDO_Faixa,Total_Bens_Log_Faixa,Total_Gasto_Campanha_Log_Faixa,cluster,SG_UF
0,160002532588,PSB,MASCULINO,SUPERIOR COMPLETO,SOLTEIRO(A),AMARELA,PR,Q1 (21 a 40),Q2 (13 a 16),Q2 (R$ 158 mil a R$ 710 mil),Q2 (R$ 3 mil a R$ 34 mil),Estabelecidos com patrimônio e campanha,PR
1,160002532589,PSB,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,PR,Q4 (58 a 85),Q2 (13 a 16),Q2 (R$ 158 mil a R$ 710 mil),Q3 (R$ 34 mil a R$ 144 mil),Estabelecidos com patrimônio e campanha,PR
2,160002532590,PSB,FEMININO,SUPERIOR COMPLETO,SOLTEIRO(A),BRANCA,PR,Q1 (21 a 40),Q2 (13 a 16),Q1 (R$ 0 a R$ 158 mil),Q4 (R$ 144 mil a R$ 3.0 mi),Estabelecidos com patrimônio e campanha,PR
3,160002532591,PSB,MASCULINO,SUPERIOR INCOMPLETO,SOLTEIRO(A),BRANCA,PR,Q1 (21 a 40),Q1 (4 a 13),Q1 (R$ 0 a R$ 158 mil),Q1 (R$ 0 a R$ 3 mil),Nível superior sem campanha,PR
4,160002532592,PSB,MASCULINO,SUPERIOR COMPLETO,DIVORCIADO(A),BRANCA,PR,Q3 (48 a 58),Q2 (13 a 16),Q2 (R$ 158 mil a R$ 710 mil),Q3 (R$ 34 mil a R$ 144 mil),Estabelecidos com patrimônio e campanha,PR


## Itens frequentes: o que aparece sozinho, com que frequência

O Apriori funciona em duas etapas. Primeiro, encontra **itemsets frequentes** — combinações de 1, 2, 3... itens que aparecem juntas em pelo menos `min_suporte` das cestas. Só depois, na segunda etapa, ele vira **regras** (`A → B`). Começamos pelos itemsets de um item só — a frequência "crua" de cada valor, antes de cruzar qualquer coisa.


In [9]:
min_suporte = 0.02  # ~22 candidatos; ajuste se quiser regras mais raras (menor) ou mais comuns (maior)

itens_frequentes = apriori(cesta, min_support=min_suporte, use_colnames=True)
print(f"{len(itens_frequentes)} itemsets frequentes (min_suporte={min_suporte})")

individuais = itens_frequentes[itens_frequentes['itemsets'].apply(lambda x: len(x) == 1)].copy()
individuais['item'] = individuais['itemsets'].apply(lambda x: next(iter(x)))
individuais['qtd_candidatos'] = (individuais['support'] * len(cesta)).round().astype(int)
individuais.sort_values('support', ascending=False)[['item', 'support', 'qtd_candidatos']].head(15)

15241 itemsets frequentes (min_suporte=0.02)


,item,support,qtd_candidatos
30,DS_COR_RACA=BRANCA,0.788720,881
53,cluster=Estabelecidos com patrimônio e campanha,0.637422,712
20,DS_GENERO=MASCULINO,0.632945,707
43,ANOS_ESTUDO_Faixa=Q2 (13 a 16),0.608774,680
25,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,0.608774,680
44,Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil),0.500448,559
27,DS_ESTADO_CIVIL=CASADO(A),0.478962,535
56,SG_UF=RS,0.410027,458
35,SG_UF_NASCIMENTO=RS,0.400179,447
42,ANOS_ESTUDO_Faixa=Q1 (4 a 13),0.391226,437


### Suporte, confiança e lift — o que cada métrica mede

Regras de associação têm o formato **"se A, então B"** (A = antecedente, B = consequente), e três números resumem o quão forte é essa relação:

- **Suporte** — fração de candidatos em que A e B aparecem juntos: `suporte(A→B) = P(A e B)`. Mede o quão comum é a combinação; suporte baixo (poucos candidatos) é terreno fértil pra coincidência.
- **Confiança** — entre os candidatos que têm A, qual fração também tem B: `confiança(A→B) = P(B | A) = suporte(A e B) / suporte(A)`. É a "taxa de acerto" da regra.
- **Lift** — o quanto a confiança da regra é maior (ou menor) do que se A e B fossem independentes: `lift(A→B) = confiança(A→B) / suporte(B)`. `lift = 1` → A e B são independentes (a regra não diz nada); `lift > 1` → aparecem juntos mais do que o esperado ao acaso; `lift < 1` → aparecem juntos menos do que o esperado.

**Um cuidado importante nesta base:** são 1.117 candidatos espalhados por 27 partidos — vários partidos têm só 10 a 20 candidatos, e combinações de 3 ou mais itens caem rapidamente para poucas dezenas de pessoas. Com grupos pequenos, bastam algumas coincidências pra um lift parecer alto, sem que isso signifique relação real nenhuma. Por isso toda tabela de regras abaixo vem com uma coluna `qtd_candidatos` — sempre olhem ela antes de confiar num lift alto.


In [10]:
regras = association_rules(itens_frequentes, metric='lift', min_threshold=1)


def mistura_escolaridade(regra):
    """True se a regra usa ao mesmo tempo DS_GRAU_INSTRUCAO e ANOS_ESTUDO_Faixa — as duas
    medem a mesma coisa, então a regra seria verdadeira por definição."""
    itens = regra['antecedents'] | regra['consequents']
    return (any(i.startswith('DS_GRAU_INSTRUCAO=') for i in itens)
            and any(i.startswith('ANOS_ESTUDO_Faixa=') for i in itens))


regras = regras[~regras.apply(mistura_escolaridade, axis=1)]
regras['qtd_candidatos'] = (regras['support'] * len(cesta)).round().astype(int)
regras = regras.sort_values('confidence', ascending=False)


def formatar_regras(df_regras):
    """Troca antecedents/consequents (frozenset) por texto legível, só pra exibição —
    os dados originais (usados pra filtrar/ordenar) continuam intactos."""
    df_fmt = df_regras.copy()
    df_fmt['antecedents'] = df_fmt['antecedents'].apply(lambda x: ', '.join(sorted(str(i) for i in x)))
    df_fmt['consequents'] = df_fmt['consequents'].apply(lambda x: ', '.join(sorted(str(i) for i in x)))
    return df_fmt


print(f"{len(regras)} regras (lift >= 1)")
formatar_regras(regras[['antecedents', 'consequents', 'qtd_candidatos', 'support', 'confidence', 'lift']].head(10))

266648 regras (lift >= 1)


,antecedents,consequents,qtd_candidatos,support,confidence,lift
161201,"IDADE_Faixa=Q2 (40 a 48), SG_UF=RS, SG_UF_NASCIMENTO=RS, Total_Bens_Log_Faixa=Q2 (R$ 158 mil a R$ 710 mil)",cluster=Estabelecidos com patrimônio e campanha,28,0.025067,1.0,1.568820
217083,"DS_ESTADO_CIVIL=CASADO(A), DS_GENERO=MASCULINO, SG_UF_NASCIMENTO=PR, Total_Bens_Log_Faixa=Q3 (R$ 710 mil a R$ 75.7 mi), Total_Gasto_Campanha_Log_Faixa=Q4 (R$ 144 mil a R$ 3.0 mi)",cluster=Estabelecidos com patrimônio e campanha,32,0.028648,1.0,1.568820
6154,"DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO, cluster=Em campanha sem patrimônio declarado",Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil),49,0.043868,1.0,1.998211
233187,"DS_GENERO=MASCULINO, IDADE_Faixa=Q4 (58 a 85), SG_UF=PR, SG_UF_NASCIMENTO=PR, Total_Gasto_Campanha_Log_Faixa=Q4 (R$ 144 mil a R$ 3.0 mi)",cluster=Estabelecidos com patrimônio e campanha,24,0.021486,1.0,1.568820
304131,"DS_ESTADO_CIVIL=CASADO(A), DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, IDADE_Faixa=Q4 (58 a 85), Total_Bens_Log_Faixa=Q3 (R$ 710 mil a R$ 75.7 mi), Total_Gasto_Campanha_Log_Faixa=Q4 (R$ 144 mil a R$ 3.0 mi)",DS_COR_RACA=BRANCA,41,0.036705,1.0,1.267877
195982,"DS_COR_RACA=BRANCA, DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, IDADE_Faixa=Q3 (48 a 58), Total_Bens_Log_Faixa=Q3 (R$ 710 mil a R$ 75.7 mi)",cluster=Estabelecidos com patrimônio e campanha,40,0.035810,1.0,1.568820
196044,"DS_COR_RACA=BRANCA, DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, IDADE_Faixa=Q3 (48 a 58), Total_Gasto_Campanha_Log_Faixa=Q4 (R$ 144 mil a R$ 3.0 mi)",cluster=Estabelecidos com patrimônio e campanha,31,0.027753,1.0,1.568820
234495,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), DS_GENERO=MASCULINO, SG_UF_NASCIMENTO=RS, Total_Gasto_Campanha_Log_Faixa=Q2 (R$ 3 mil a R$ 34 mil), cluster=Em campanha sem patrimônio declarado",Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil),24,0.021486,1.0,1.998211
78409,"DS_ESTADO_CIVIL=CASADO(A), DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, Total_Bens_Log_Faixa=Q3 (R$ 710 mil a R$ 75.7 mi)",cluster=Estabelecidos com patrimônio e campanha,136,0.121755,1.0,1.568820
195600,"DS_COR_RACA=BRANCA, DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, IDADE_Faixa=Q2 (40 a 48), Total_Gasto_Campanha_Log_Faixa=Q4 (R$ 144 mil a R$ 3.0 mi)",cluster=Estabelecidos com patrimônio e campanha,31,0.027753,1.0,1.568820


### Cuidado: suporte pequeno infla o lift

Vamos ver isso na prática: as regras de maior lift da base inteira, sem nenhum filtro de suporte, comparadas às regras de maior lift **exigindo pelo menos 10 candidatos** por trás delas.


In [11]:
regras_1_consequente = regras[regras['consequents'].apply(lambda x: len(x) == 1)].copy()
colunas_exibir = ['antecedents', 'consequents', 'qtd_candidatos', 'confidence', 'lift']

print("--- Maior lift, sem filtro de suporte ---")
display(formatar_regras(regras_1_consequente.sort_values('lift', ascending=False)[colunas_exibir].head(5)))

print("--- Maior lift, exigindo qtd_candidatos >= 10 ---")
display(formatar_regras(regras_1_consequente[regras_1_consequente['qtd_candidatos'] >= 10]
        .sort_values('lift', ascending=False)[colunas_exibir].head(5)))


--- Maior lift, sem filtro de suporte ---


,antecedents,consequents,qtd_candidatos,confidence,lift
56914,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), IDADE_Faixa=Q4 (58 a 85), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,31,0.861111,10.807428
163144,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), IDADE_Faixa=Q4 (58 a 85), Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,23,0.851852,10.691219
157306,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), DS_COR_RACA=BRANCA, IDADE_Faixa=Q4 (58 a 85), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,23,0.851852,10.691219
40474,"DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO, Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Nível superior sem campanha,25,0.781250,9.283577
56563,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), IDADE_Faixa=Q3 (48 a 58), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,25,0.714286,8.964687


--- Maior lift, exigindo qtd_candidatos >= 10 ---


,antecedents,consequents,qtd_candidatos,confidence,lift
56914,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), IDADE_Faixa=Q4 (58 a 85), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,31,0.861111,10.807428
163144,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), IDADE_Faixa=Q4 (58 a 85), Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,23,0.851852,10.691219
157306,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), DS_COR_RACA=BRANCA, IDADE_Faixa=Q4 (58 a 85), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,23,0.851852,10.691219
40474,"DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO, Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Nível superior sem campanha,25,0.781250,9.283577
56563,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), IDADE_Faixa=Q3 (48 a 58), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,25,0.714286,8.964687


Com `min_suporte = 0.02`, toda regra que sai do Apriori já tem pelo menos ~22 candidatos por trás — por isso as duas listas acima coincidem: o piso de suporte da própria mineração já fez o papel do filtro de 10 candidatos. Se baixarem o `min_suporte` (ex.: 0,005, ~6 candidatos), vale repetir a comparação: é nas regras sustentadas por pouquíssimos candidatos que o lift costuma inflar.

As regras de maior lift (≈ 10) são combinações das **faixas das variáveis do modelo** — escolaridade até 13 anos + idade de 58 anos ou mais + gasto na faixa mais baixa → `cluster=Base popular sem campanha`. Elas estão certas, mas são quase circulares: o cluster foi construído com essas mesmas variáveis. Por isso, mais abaixo, separamos as regras que chegam ao cluster **sem** usar as variáveis do modelo.

### Filtrando pra regras que valem a pena olhar

In [12]:
suporte_pratico = 0.03    # ~34 candidatos
confianca_pratica = 0.5

regras_simples = regras[
    regras['antecedents'].apply(lambda x: len(x) <= 2) & regras['consequents'].apply(lambda x: len(x) == 1)
]
regras_confiaveis = regras_simples[
    (regras_simples['support'] >= suporte_pratico) & (regras_simples['confidence'] >= confianca_pratica)
]

print(f"{len(regras_confiaveis)} regras com suporte >= {suporte_pratico} e confiança >= {confianca_pratica}")
formatar_regras(regras_confiaveis.sort_values('confidence', ascending=False)[colunas_exibir].head(15))

1751 regras com suporte >= 0.03 e confiança >= 0.5


,antecedents,consequents,qtd_candidatos,confidence,lift
6154,"DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO, cluster=Em campanha sem patrimônio declarado",Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil),49,1.0,1.998211
6668,"DS_ESTADO_CIVIL=CASADO(A), cluster=Base popular sem campanha",ANOS_ESTUDO_Faixa=Q1 (4 a 13),35,1.0,2.556064
5765,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, cluster=Nível superior sem campanha",Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil),43,1.0,3.989286
5740,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, Total_Bens_Log_Faixa=Q3 (R$ 710 mil a R$ 75.7 mi)",cluster=Estabelecidos com patrimônio e campanha,219,1.0,1.568820
7341,"DS_ESTADO_CIVIL=SOLTEIRO(A), SG_UF_NASCIMENTO=SC",SG_UF=SC,51,1.0,4.835498
7627,"DS_ESTADO_CIVIL=SOLTEIRO(A), cluster=Nível superior sem campanha",Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil),50,1.0,3.989286
7620,"DS_ESTADO_CIVIL=SOLTEIRO(A), Total_Bens_Log_Faixa=Q3 (R$ 710 mil a R$ 75.7 mi)",cluster=Estabelecidos com patrimônio e campanha,43,1.0,1.568820
7589,"DS_ESTADO_CIVIL=SOLTEIRO(A), cluster=Nível superior sem campanha",Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil),50,1.0,1.998211
5709,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, Total_Bens_Log_Faixa=Q2 (R$ 158 mil a R$ 710 mil)",cluster=Estabelecidos com patrimônio e campanha,203,1.0,1.568820
5679,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, cluster=Em campanha sem patrimônio declarado",Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil),83,1.0,1.998211


## O que prediz o cluster de um candidato?

Filtramos as regras cujo **consequente é um único item de cluster** — ou seja, "se o candidato tem tais características, ele cai no cluster X". É a mesma pergunta que a Fase 2 respondeu via ficha técnica, agora vista pelo ângulo de combinações de atributos categóricos.

Um cuidado extra aqui: os quatro clusters são **desbalanceados** (A = 64%, B = 20%, C = 8%, D = 8%). Ordenar só por confiança faria a tabela inteira virar uma lista de regras "→ A", fácil de acertar por ser a maioria, mas com lift baixo (no máximo 1/0,64 ≈ 1,6). E um piso fixo de confiança (ex.: 0,5) teria o efeito oposto: os clusters pequenos quase não aparecem. Por isso pegamos as regras de **maior lift dentro de cada cluster**, separadamente — assim os quatro aparecem, e dá pra comparar a força relativa de cada um.

Como as faixas de patrimônio e de gasto também estão na cesta, as regras com essas faixas no antecedente são quase uma "releitura" da própria clusterização (o cluster foi construído com essas variáveis). As regras mais informativas são as que chegam ao cluster a partir de variáveis que **não** entraram no modelo — gênero, cor/raça, partido, estado civil, UF.

In [13]:
def regras_para_consequente(regras_completas, prefixo, n_por_valor=6, min_suporte=min_suporte, max_antecedentes=2):
    """
    Entre as regras cujo único item consequente comece com `prefixo` (ex.: 'cluster='),
    devolve as `n_por_valor` de MAIOR LIFT pra CADA valor distinto do consequente — sem piso
    de confiança fixo, porque um valor raro (poucos candidatos) não sustenta regras de
    confiança alta mesmo quando o lift é forte; ordenar só por confiança deixaria os valores
    mais comuns dominando a tabela inteira (ver célula anterior).
    """
    candidatas = regras_completas[
        regras_completas['consequents'].apply(lambda x: len(x) == 1 and next(iter(x)).startswith(prefixo))
        & regras_completas['antecedents'].apply(lambda x: len(x) <= max_antecedentes)
        & (regras_completas['support'] >= min_suporte)
    ].copy()
    candidatas['alvo'] = candidatas['consequents'].apply(lambda x: next(iter(x)))
    resultado = (candidatas.sort_values('lift', ascending=False)
                            .groupby('alvo', group_keys=False)
                            .head(n_por_valor)
                            .sort_values(['alvo', 'lift'], ascending=[True, False]))
    return resultado.drop(columns='alvo')


regras_cluster = regras_para_consequente(regras, 'cluster=')
print(f"{len(regras_cluster)} regras (até 6 por cluster, ordenadas por lift dentro de cada um)")
formatar_regras(regras_cluster[colunas_exibir])


24 regras (até 6 por cluster, ordenadas por lift dentro de cada um)


,antecedents,consequents,qtd_candidatos,confidence,lift
10314,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,85,0.537975,6.751885
10201,"IDADE_Faixa=Q4 (58 a 85), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,31,0.525424,6.594363
4860,"DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO, Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,36,0.514286,6.454575
10805,"SG_UF=RS, Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,49,0.392000,4.919820
9196,"SG_UF_NASCIMENTO=RS, Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,45,0.384615,4.827139
10010,"IDADE_Faixa=Q3 (48 a 58), Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil)",cluster=Base popular sem campanha,25,0.378788,4.754001
10614,"Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil), Total_Gasto_Campanha_Log_Faixa=Q2 (R$ 3 mil a R$ 34 mil)",cluster=Em campanha sem patrimônio declarado,102,0.645570,3.248204
4877,"DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO, Total_Gasto_Campanha_Log_Faixa=Q2 (R$ 3 mil a R$ 34 mil)",cluster=Em campanha sem patrimônio declarado,24,0.585366,2.945287
2230,"SG_PARTIDO=PSOL, Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil)",cluster=Em campanha sem patrimônio declarado,23,0.575000,2.893131
10335,"ANOS_ESTUDO_Faixa=Q1 (4 a 13), Total_Gasto_Campanha_Log_Faixa=Q2 (R$ 3 mil a R$ 34 mil)",cluster=Em campanha sem patrimônio declarado,68,0.552846,2.781660


**Leitura.** Com as faixas das variáveis do modelo no antecedente, as regras reconstroem os clusters quase por definição — é uma checagem de consistência (as faixas por quartil contam a mesma história que os centróides), não uma descoberta:

- **Base popular sem campanha** ← escolaridade até 13 anos + gasto até R$ 3 mil (lift 6,8, 85 candidatos), ou 58 anos ou mais + gasto até R$ 3 mil (lift 6,6).
- **Nível superior sem campanha** ← superior incompleto + gasto até R$ 3 mil (lift 8,3, 25 candidatos), ou até 40 anos + gasto até R$ 3 mil (lift 5,6).
- **Em campanha sem patrimônio declarado** ← bens até R$ 158 mil + gasto de R$ 3 mil a R$ 34 mil (lift 3,2, 102 candidatos).
- **Estabelecidos com patrimônio e campanha** ← bens acima de R$ 710 mil + qualquer outro item (confiança 1). O lift fica em 1,57, que é o teto possível para um cluster com 64% da base (1 / 0,64).

A pergunta mais interessante é o que prediz o cluster **sem** usar as variáveis do modelo — ou seja, só com gênero, cor/raça, estado civil, partido e UF:

In [14]:
variaveis_do_modelo = tuple(f'{c}=' for c in colunas_faixa) + ('DS_GRAU_INSTRUCAO=',)

regras_sem_modelo = regras[
    regras['antecedents'].apply(lambda x: not any(str(i).startswith(variaveis_do_modelo) for i in x))
]
regras_cluster_externas = regras_para_consequente(regras_sem_modelo, 'cluster=')
print(f"{len(regras_cluster_externas)} regras sem variáveis do modelo no antecedente (até 6 por cluster)")
formatar_regras(regras_cluster_externas[colunas_exibir])

24 regras sem variáveis do modelo no antecedente (até 6 por cluster)


,antecedents,consequents,qtd_candidatos,confidence,lift
4034,"DS_GENERO=MASCULINO, SG_UF_NASCIMENTO=RS",cluster=Base popular sem campanha,33,0.114583,1.438085
4487,"DS_GENERO=MASCULINO, SG_UF=RS",cluster=Base popular sem campanha,34,0.114478,1.436765
8316,"DS_COR_RACA=BRANCA, SG_UF=RS",cluster=Base popular sem campanha,40,0.109890,1.379183
786,SG_UF=RS,cluster=Base popular sem campanha,50,0.109170,1.370149
9254,"SG_UF=RS, SG_UF_NASCIMENTO=RS",cluster=Base popular sem campanha,44,0.106538,1.337106
619,SG_UF_NASCIMENTO=RS,cluster=Base popular sem campanha,46,0.102908,1.291557
7230,"DS_COR_RACA=PRETA, DS_ESTADO_CIVIL=SOLTEIRO(A)",cluster=Em campanha sem patrimônio declarado,28,0.459016,2.309555
8507,"DS_COR_RACA=PRETA, SG_UF_NASCIMENTO=RS",cluster=Em campanha sem patrimônio declarado,23,0.450980,2.269122
2844,"DS_COR_RACA=PRETA, DS_GENERO=FEMININO",cluster=Em campanha sem patrimônio declarado,25,0.423729,2.132005
238,SG_PARTIDO=PSOL,cluster=Em campanha sem patrimônio declarado,23,0.410714,2.066522


**Leitura (sem as variáveis do modelo).**

- **Em campanha sem patrimônio declarado** é o cluster com as regras externas mais fortes: `DS_COR_RACA=PRETA` → B (lift 2,0: 39% dos candidatos pretos estão nesse cluster, contra 20% da base), `PRETA + SOLTEIRO(A)` (lift 2,3), `PRETA + FEMININO` (lift 2,1) e `SG_PARTIDO=PSOL` (lift 2,1).
- **Estabelecidos com patrimônio e campanha** ← partidos grandes e com estrutura: PP, PT, PODE, PDT (no RS) e NOVO, com confiança de 0,89 a 0,97 e lift de 1,4 a 1,5 (perto do teto de 1,57). São legendas de todo o espectro ideológico: ter patrimônio e campanha estruturada se associa mais ao **tamanho do partido** do que à ideologia.
- **Base popular sem campanha** ← `SG_UF=RS` (lift 1,4: 11% dos candidatos gaúchos estão nesse cluster, contra 8% da base).
- **Nível superior sem campanha** ← `SOLTEIRO(A)` (lift 1,5).

Os lifts externos são moderados (no máximo 2,3): gênero, raça e partido não *definem* os clusters, mas deslocam as proporções de forma consistente — e isso aparece mesmo sem essas variáveis terem entrado na clusterização.

### Visualizando como grafo

Uma tabela com dezenas de regras é difícil de escanear — um grafo interativo ajuda a ver de relance quais itens aparecem mais como "explicação" (antecedentes) e quais aparecem mais como "resultado" (consequentes). Cada item vira um nó (`SG_PARTIDO=PCO`, `cluster=D`, ...); cada regra vira uma seta do antecedente pro consequente, com espessura proporcional à confiança. Os nós que começam com `cluster=` ficam destacados em vermelho e maiores, pra serem fáceis de achar no meio dos outros itens. Passe o mouse sobre uma seta pra ver confiança, lift e quantos candidatos sustentam aquela regra — e arraste os nós se a rede ficar embaralhada.


In [15]:
def grafo_regras_html(regras_plot, arquivo_html, titulo, prefixo_destaque='cluster='):
    """
    Desenha as regras como uma rede interativa (pyvis) e grava em `arquivo_html`. Cada REGRA
    vira um nó de antecedente e um nó de consequente (quando o antecedente tem mais de um
    item, eles aparecem juntos no mesmo nó, separados por "+") ligados por UMA única aresta
    — nunca uma aresta por item isolado, pra não sugerir que um item sozinho sustenta a
    confiança/lift de uma regra que na verdade depende da combinação inteira (era isso que
    causava setas duplicadas entre o mesmo par de nós). Nós que contêm algum item começando
    com `prefixo_destaque` (por padrão, os clusters da Fase 2) ficam destacados em vermelho e
    maiores. Espessura da aresta = confiança da regra; passe o mouse pra ver confiança, lift
    e quantos candidatos a sustentam. Os controles de física (embaixo do grafo) deixam
    ajustar a repulsão/gravidade entre os nós ao vivo.

    Retorna um IFrame carregando o arquivo — se aparecer em branco/preto aqui embaixo (alguns
    ambientes bloqueiam o JavaScript de saídas de célula em notebooks "não confiáveis"), abra
    `arquivo_html` direto no navegador; o grafo é o mesmo.
    """
    print(titulo)
    net = Network(height='600px', width='100%', bgcolor='#222222', font_color='white',
                  notebook=True, directed=True, cdn_resources='in_line')

    nos_adicionados = set()
    for _, row in regras_plot.iterrows():
        antecedente = ' + '.join(sorted(str(item) for item in row['antecedents']))
        consequente = ' + '.join(sorted(str(item) for item in row['consequents']))

        for label, itemset in [(antecedente, row['antecedents']), (consequente, row['consequents'])]:
            if label in nos_adicionados:
                continue
            destaque = any(str(item).startswith(prefixo_destaque) for item in itemset)
            net.add_node(
                label, label, title=label,
                color='#e41a1c' if destaque else '#1f78b4',
                size=30 if destaque else 12,
                font={'size': 22 if destaque else 14},
            )
            nos_adicionados.add(label)

        titulo_aresta = (f"Confiança: {row['confidence']:.2f} | Lift: {row['lift']:.2f} | "
                          f"{row['qtd_candidatos']} candidatos")
        net.add_edge(antecedente, consequente, value=row['confidence'], title=titulo_aresta)

    net.show_buttons(filter_=['physics'])
    net.save_graph(arquivo_html)
    print(f"Grafo salvo em: {arquivo_html} — se não aparecer abaixo, abra esse arquivo no navegador.")
    return IFrame(arquivo_html, width='100%', height=750)


grafo_regras_html(regras_cluster.head(20), 'grafo_regras_cluster.html', 'O que prediz o cluster de um candidato?')


O que prediz o cluster de um candidato?
Grafo salvo em: grafo_regras_cluster.html — se não aparecer abaixo, abra esse arquivo no navegador.


## O que caracteriza quem já é de um cluster?

A seção anterior fixou o **consequente** (`cluster=X`) e perguntou o que o prediz. Agora invertemos: fixamos o **antecedente** — "dado que o candidato é do cluster X, ..." — e olhamos pra que outros itens aparecem como consequente com lift alto. É uma pergunta diferente, não a mesma coisa ao contrário: uma regra `A → B` de lift alto não garante que `B → A` também tenha lift alto (confiança e suporte de cada lado são diferentes), então vale olhar as duas direções separadamente.


In [16]:
def regras_para_antecedente(regras_completas, item_fixo, n=8, min_suporte=min_suporte, max_consequentes=1):
    """
    Entre as regras cujo antecedente é EXATAMENTE {item_fixo} (um único item), devolve as
    `n` de maior lift. Direção oposta de `regras_para_consequente`: lá fixamos o "efeito"
    (o consequente) e procurávamos o que o prediz; aqui fixamos a "causa" (o antecedente) e
    olhamos o que ela prediz.
    """
    candidatas = regras_completas[
        regras_completas['antecedents'].apply(lambda x: len(x) == 1 and next(iter(x)) == item_fixo)
        & regras_completas['consequents'].apply(lambda x: len(x) <= max_consequentes)
        & (regras_completas['support'] >= min_suporte)
    ]
    return candidatas.sort_values('lift', ascending=False).head(n)


regras_por_cluster_fixo = {
    f'cluster={nome}': regras_para_antecedente(regras, f'cluster={nome}')
    for nome in NOMES_CLUSTER_BISECTING.values()
}

for item_fixo, subset in regras_por_cluster_fixo.items():
    print(f"--- {item_fixo} ({len(subset)} regras) ---")
    display(formatar_regras(subset[colunas_exibir]))

--- cluster=Estabelecidos com patrimônio e campanha (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
757,cluster=Estabelecidos com patrimônio e campanha,Total_Bens_Log_Faixa=Q3 (R$ 710 mil a R$ 75.7 mi),275,0.386236,1.551891
750,cluster=Estabelecidos com patrimônio e campanha,Total_Bens_Log_Faixa=Q2 (R$ 158 mil a R$ 710 mil),258,0.362360,1.445556
781,cluster=Estabelecidos com patrimônio e campanha,Total_Gasto_Campanha_Log_Faixa=Q4 (R$ 144 mil a R$ 3.0 mi),257,0.360955,1.445114
167,cluster=Estabelecidos com patrimônio e campanha,SG_PARTIDO=PP,33,0.046348,1.362397
255,cluster=Estabelecidos com patrimônio e campanha,SG_PARTIDO=PT,47,0.066011,1.316688
155,cluster=Estabelecidos com patrimônio e campanha,SG_PARTIDO=PODE,65,0.091292,1.307350
130,cluster=Estabelecidos com patrimônio e campanha,SG_PARTIDO=PL,67,0.094101,1.297666
729,cluster=Estabelecidos com patrimônio e campanha,ANOS_ESTUDO_Faixa=Q2 (13 a 16),554,0.778090,1.278127


--- cluster=Em campanha sem patrimônio declarado (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
239,cluster=Em campanha sem patrimônio declarado,SG_PARTIDO=PSOL,23,0.103604,2.066522
741,cluster=Em campanha sem patrimônio declarado,Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil),220,0.990991,1.980209
583,cluster=Em campanha sem patrimônio declarado,DS_COR_RACA=PRETA,41,0.184685,1.964693
771,cluster=Em campanha sem patrimônio declarado,Total_Gasto_Campanha_Log_Faixa=Q2 (R$ 3 mil a R$ 34 mil),103,0.463964,1.857519
380,cluster=Em campanha sem patrimônio declarado,DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO,60,0.270270,1.705604
714,cluster=Em campanha sem patrimônio declarado,ANOS_ESTUDO_Faixa=Q1 (4 a 13),139,0.626126,1.600418
442,cluster=Em campanha sem patrimônio declarado,DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO,49,0.220721,1.512546
566,cluster=Em campanha sem patrimônio declarado,DS_COR_RACA=PARDA,36,0.162162,1.496985


--- cluster=Nível superior sem campanha (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
765,cluster=Nível superior sem campanha,Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil),94,1.000000,3.989286
743,cluster=Nível superior sem campanha,Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil),94,1.000000,1.998211
445,cluster=Nível superior sem campanha,DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO,25,0.265957,1.822543
655,cluster=Nível superior sem campanha,IDADE_Faixa=Q1 (21 a 40),45,0.478723,1.770643
525,cluster=Nível superior sem campanha,DS_ESTADO_CIVIL=SOLTEIRO(A),50,0.531915,1.489095
716,cluster=Nível superior sem campanha,ANOS_ESTUDO_Faixa=Q1 (4 a 13),51,0.542553,1.386801
799,cluster=Nível superior sem campanha,SG_UF=RS,44,0.468085,1.141596
305,cluster=Nível superior sem campanha,DS_GENERO=FEMININO,38,0.404255,1.101349


--- cluster=Base popular sem campanha (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
762,cluster=Base popular sem campanha,Total_Gasto_Campanha_Log_Faixa=Q1 (R$ 0 a R$ 3 mil),85,0.955056,3.809992
712,cluster=Base popular sem campanha,ANOS_ESTUDO_Faixa=Q1 (4 a 13),89,1.000000,2.556064
378,cluster=Base popular sem campanha,DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO,36,0.404494,2.552657
699,cluster=Base popular sem campanha,IDADE_Faixa=Q4 (58 a 85),34,0.382022,1.610261
738,cluster=Base popular sem campanha,Total_Bens_Log_Faixa=Q1 (R$ 0 a R$ 158 mil),66,0.741573,1.481819
787,cluster=Base popular sem campanha,SG_UF=RS,50,0.561798,1.370149
618,cluster=Base popular sem campanha,SG_UF_NASCIMENTO=RS,46,0.516854,1.291557
682,cluster=Base popular sem campanha,IDADE_Faixa=Q3 (48 a 58),26,0.292135,1.199686


**Leitura (antecedente fixo).**

- **Estabelecidos com patrimônio e campanha** → bens acima de R$ 710 mil e gasto acima de R$ 144 mil (lift ≈ 1,5), superior completo (78%), e os partidos PP, PT, PODE e PL.
- **Em campanha sem patrimônio declarado** → PSOL (lift 2,1), cor preta (2,0) e parda (1,5), ensino médio completo (1,7) e superior incompleto (1,5). 99% têm bens até R$ 158 mil, e o gasto se concentra entre R$ 3 mil e R$ 34 mil: campanha ativa, mas modesta.
- **Nível superior sem campanha** → gasto até R$ 3 mil (100%), bens até R$ 158 mil (100%), superior incompleto (lift 1,8), até 40 anos (1,8) e solteiro(a) (1,5).
- **Base popular sem campanha** → escolaridade até 13 anos (100%), ensino médio completo (lift 2,6), 58 anos ou mais (1,6) e RS (1,4).

### As regras confirmam os nomes dos clusters?

| Cluster | Nome | O que as regras mostram | Confirma? |
|---|---|---|---|
| A | Estabelecidos com patrimônio e campanha | bens e gasto nas faixas mais altas, superior completo, partidos grandes | **Sim** |
| B | Em campanha sem patrimônio declarado | 99% com bens na faixa mais baixa, gasto entre R$ 3 mil e R$ 144 mil; associado a PSOL e a cor preta/parda | **Sim** — e acrescenta um recorte racial e partidário que o nome não diz |
| C | Nível superior sem campanha | gasto e bens na faixa mais baixa (100%), superior incompleto, até 40 anos, solteiros | **Sim, com ressalva:** o nível superior é alto em relação ao D, não à base inteira; o traço mais forte é superior *incompleto*, então é um grupo mais jovem e ainda em formação |
| D | Base popular sem campanha | escolaridade até 13 anos (100%), ensino médio, 58 anos ou mais, RS | **Sim** |

Uma primeira versão dos nomes chamava o C de "Escolarizados"; as regras mostraram que isso só valia na comparação com o D, e por isso o nome foi ajustado.

### Visualizando como grafo (antecedente fixo)

Mesmo grafo interativo de antes, só que agora as três "causas" (os clusters) são o ponto de partida fixo, e os itens que aparecem como consequência é que variam.


In [17]:
regras_antecedente_fixo = pd.concat(regras_por_cluster_fixo.values())
grafo_regras_html(regras_antecedente_fixo, 'grafo_regras_antecedente_fixo.html', 'O que caracteriza quem já é de um cluster?')


O que caracteriza quem já é de um cluster?
Grafo salvo em: grafo_regras_antecedente_fixo.html — se não aparecer abaixo, abra esse arquivo no navegador.


## E quando a eleição já tiver ocorrido?

Esta seção só roda depois que a Fase 0 for rodada de novo com o resultado oficial (`DS_SIT_TOT_TURNO` deixa de vir toda com o mesmo valor `#NULO`) — por enquanto, ela se anuncia e não faz nada. Repare que não precisamos escrever nenhum código novo: como `DS_SIT_TOT_TURNO` já entrou na cesta lá na preparação (quando aplicável), a mesma `regras_para_consequente` e o mesmo `grafo_regras_html` servem pra essa pergunta também.


In [18]:
if usar_resultado_eleicao:
    regras_eleicao = regras_para_consequente(regras, 'DS_SIT_TOT_TURNO=')
    print(f"{len(regras_eleicao)} regras apontando para o resultado da eleição")
    display(formatar_regras(regras_eleicao[colunas_exibir].head(15)))
    grafo_regras_html(regras_eleicao.head(20), 'grafo_regras_eleicao.html', 'O que prediz ser eleito?', prefixo_destaque='DS_SIT_TOT_TURNO=')
else:
    print("DS_SIT_TOT_TURNO ainda não tem informação real (a eleição de 2026 não ocorreu) — "
          "essa seção fica pronta pra usar assim que a Fase 0 for rodada de novo com o resultado oficial. "
          "Por enquanto, pulamos.")


DS_SIT_TOT_TURNO ainda não tem informação real (a eleição de 2026 não ocorreu) — essa seção fica pronta pra usar assim que a Fase 0 for rodada de novo com o resultado oficial. Por enquanto, pulamos.


---
## Fechamento e próximos passos

Resumindo o roteiro: transformamos cada candidato numa "cesta" de atributos categóricos (partido, gênero, escolaridade, estado civil, cor/raça, UF, UF de nascimento, as faixas por quartil das 4 variáveis da clusterização e o cluster da Fase 2), mineramos itemsets frequentes com o Apriori, geramos regras `A → B` com suporte/confiança/lift e olhamos o cluster nas duas posições — como consequente ("o que prediz o cluster?") e como antecedente ("o que caracteriza quem é do cluster?").

Principais achados:
- as faixas das variáveis do modelo reconstroem os 4 clusters com lift alto, o que confirma a consistência da clusterização;
- sem as variáveis do modelo, a associação mais forte é entre **cor/raça preta, PSOL e o cluster "Em campanha sem patrimônio declarado"** (lift ≈ 2), e entre **partidos grandes e o cluster com patrimônio e campanha**;
- as regras confirmam os nomes de A, B e D, e levaram a ajustar o nome do C.

**Exercícios sugeridos:**
- Agrupar `DS_OCUPACAO` em categorias maiores (ex.: "política", "direito", "saúde", "educação", "empresário"...) e incluir na cesta.
- Variar `min_suporte` (célula da seção "Itens frequentes") e ver como o número de regras muda.
- Replicar para `SENADOR` ou `DEPUTADO ESTADUAL` — só muda a célula de configuração, do mesmo jeito que nas fases anteriores.
- Depois que a eleição ocorrer: rodar a Fase 0 de novo, e conferir se a seção "E quando a eleição já tiver ocorrido?" acima encontra regras fortes ligadas a `DS_SIT_TOT_TURNO=ELEITO`.

Próxima etapa do pipeline: `04_deteccao_anomalias.ipynb`.